In [9]:
import pandas as pd
from marginal_emissions.utils.helper import test_stationarity

def get_diffs(df : pd.DataFrame) -> pd.DataFrame:
    delta_df = df - df.shift(1)
    delta_df = delta_df[1:]
    return delta_df

data_dict = {
    'f_hertz_2023': pd.read_csv('../data/processed/analysis_f_hertz_2023_15min_utc_202212232300_202401010000.csv', index_col=0),
    'f_hertz_2024': pd.read_csv('../data/processed/analysis_f_hertz_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'amprion_2023': pd.read_csv('../data/processed/analysis_amprion_2023_15min_utc_202212232300_202401010000.csv', index_col=0),
    'amprion_2024': pd.read_csv('../data/processed/analysis_amprion_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'tennet_2023': pd.read_csv('../data/processed/analysis_tennet_2023_15min_utc_202212232300_202401010000.csv', index_col=0),
    'tennet_2024': pd.read_csv('../data/processed/analysis_tennet_2024_15min_utc_202312232300_202501010000.csv', index_col=0),
    'transnet_bw_2023': pd.read_csv('../data/processed/analysis_transnet_bw_2023_15min_utc_202212232300_202401010000.csv', index_col=0),
    'transnet_bw_2024': pd.read_csv('../data/processed/analysis_transnet_bw_2024_15min_utc_202312232300_202501010000.csv', index_col=0)
}

for area, df in data_dict.items():
    data_dict_diff = {
        area: get_diffs(df),
    }

for df in data_dict.values():
    df.index = pd.to_datetime(df.index, format='ISO8601')

In [8]:
for area, df in data_dict.items():
    test_stationarity(df['total_generation'], f'{area} total_generation')
    test_stationarity(df['total_emissions'], f'{area} total_emissions')

[INFO][2026-02-28 14:34:54][helper.py] Performing stationarity test for f_hertz_2023 total_generation
ADF-Statistik:  -8.2741
p-Wert:         0.0000
Verwendete Lags: 47
Anzahl Beobachtungen: 35765
Kritische Werte:
   1%: -3.4305
   5%: -2.8616
   10%: -2.5668
----------------------------------------
Ergebnis: p-Wert < 0.05. Die Nullhypothese wird abgelehnt.
-> Die Zeitreihe ist STATIONÄR (ohne stochastischen Trend).


[INFO][2026-02-28 14:34:55][helper.py] Performing stationarity test for f_hertz_2023 total_emissions
ADF-Statistik:  -8.5041
p-Wert:         0.0000
Verwendete Lags: 53
Anzahl Beobachtungen: 35759
Kritische Werte:
   1%: -3.4305
   5%: -2.8616
   10%: -2.5668
----------------------------------------
Ergebnis: p-Wert < 0.05. Die Nullhypothese wird abgelehnt.
-> Die Zeitreihe ist STATIONÄR (ohne stochastischen Trend).


[INFO][2026-02-28 14:34:57][helper.py] Performing stationarity test for f_hertz_2024 total_generation
ADF-Statistik:  -8.1047
p-Wert:         0.0000
Verwende

In [12]:
emissions = pd.read_csv('../data/interim/emissions_germany_utc_202212232200_202501012200.csv', index_col=0)
emissions.index = pd.to_datetime(emissions.index, format='ISO8601')

In [14]:
test_stationarity(emissions['total_emissions'], 'Germany total_emissions')

[INFO][2026-02-28 15:33:58][helper.py] Performing stationarity test for Germany total_emissions
ADF-Statistik:  -8.4215
p-Wert:         0.0000
Verwendete Lags: 44
Anzahl Beobachtungen: 17716
Kritische Werte:
   1%: -3.4307
   5%: -2.8617
   10%: -2.5669
----------------------------------------
Ergebnis: p-Wert < 0.05. Die Nullhypothese wird abgelehnt.
-> Die Zeitreihe ist STATIONÄR (ohne stochastischen Trend).




(np.float64(-8.421536648520734),
 np.float64(1.9774757889499868e-13),
 44,
 17716,
 {'1%': np.float64(-3.430719171808431),
  '5%': np.float64(-2.8617031598058578),
  '10%': np.float64(-2.5668568457076786)},
 np.float64(273718.0098098661))

In [15]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# Wir testen, ob in den ersten 10 Lags eine Autokorrelation vorliegt
# return_df=True gibt die Ergebnisse schön als Tabelle aus
ljung_box_results = acorr_ljungbox(emissions['total_emissions'], lags=[10], return_df=True)

print(ljung_box_results)

         lb_stat  lb_pvalue
10  138575.51505        0.0


In [18]:
for area, df in data_dict.items():
    print(f"\nArea: {area}")
    print("Generation:")
    ljung_box_results = acorr_ljungbox(df['total_generation'], lags=[10], return_df=True)
    print(ljung_box_results)
    print("Emissions:")
    ljung_box_results = acorr_ljungbox(df['total_emissions'], lags=[10], return_df=True)
    print(ljung_box_results)


Area: f_hertz_2023
Generation:
          lb_stat  lb_pvalue
10  344097.998574        0.0
Emissions:
          lb_stat  lb_pvalue
10  342500.505339        0.0

Area: f_hertz_2024
Generation:
         lb_stat  lb_pvalue
10  342598.53385        0.0
Emissions:
          lb_stat  lb_pvalue
10  338401.780155        0.0

Area: amprion_2023
Generation:
          lb_stat  lb_pvalue
10  339558.614331        0.0
Emissions:
         lb_stat  lb_pvalue
10  344128.52599        0.0

Area: amprion_2024
Generation:
          lb_stat  lb_pvalue
10  331372.394004        0.0
Emissions:
          lb_stat  lb_pvalue
10  333615.533533        0.0

Area: tennet_2023
Generation:
          lb_stat  lb_pvalue
10  331097.962037        0.0
Emissions:
          lb_stat  lb_pvalue
10  335923.223587        0.0

Area: tennet_2024
Generation:
          lb_stat  lb_pvalue
10  324733.578859        0.0
Emissions:
         lb_stat  lb_pvalue
10  326831.52235        0.0

Area: transnet_bw_2023
Generation:
          lb_stat 

In [ ]:
Macht die Verwendung des AIC in meinem Fall Sinn, oder ist eine andere Methode zur Modellauswahl besser geeignet?